# Rust CUDA kernels in `%%rust` cells

Write a GPU kernel in Rust with [cuda-oxide](https://github.com/NVlabs/cuda-oxide), compile it inside a `%%rust` cell, and launch it on Colab's T4, without leaving the notebook.

**Needs a T4 runtime** (Runtime > Change runtime type > T4 GPU). Setup takes about 1.5 minutes, the first `%%rust` cell that pulls in cuda-oxide another 1.5 minutes; after that a kernel compiles in seconds. Only Tesla T4 (sm_75) on Colab's Ubuntu 24.04 image is validated; the setup script refuses anything else rather than guess.

What the setup installs: a minimal CUDA 13.4 toolkit (740 MB, not the full 4 GB metapackage), clang-21, the nightly Rust that cuda-oxide pins, an exact cuda-oxide checkout, a prebuilt codegen backend, and a small loader crate. Everything goes under `~/.cache/colab-rust`, nothing into `/content`.


## 1. Check the runtime


In [ ]:
!nvidia-smi --query-gpu=name,driver_version --format=csv,noheader
!grep PRETTY_NAME /etc/os-release


## 2. Install

`setup-cuda-oxide.sh` runs `setup.sh` first (Rust, evcxr, the `%%rust` magic) if this session does not have it yet.


In [ ]:
!curl -fsSL -o setup-cuda-oxide.sh https://raw.githubusercontent.com/xavierforge/colab-rust/main/setup-cuda-oxide.sh
!bash setup-cuda-oxide.sh
%load_ext colab_rust


## 3. Point `%%rust` at the nightly and the backend

This is what `cargo oxide run` sets up behind the scenes. `RUSTFLAGS` is given in full because evcxr applies `:build_env` after its own flags, so this replaces them; `-Cprefer-dynamic` is evcxr's own and has to stay. The three `:dep` lines pull in cuda-oxide's crates and the loader; this is the one slow cell (about 90 seconds, once per session).


In [ ]:
%%rust
:toolchain nightly-2026-08-28
:opt 3
:build_env CARGO_INCREMENTAL=0
:build_env CUDA_HOME=/usr/local/cuda-13
:build_env CUDA_TOOLKIT_PATH=/usr/local/cuda-13
:build_env CUDA_OXIDE_TARGET=sm_75
:build_env CUDA_OXIDE_LLC=/opt/colab-rust/cuda-oxide/bin/llc
:build_env RUSTFLAGS=-Cprefer-dynamic -Zcodegen-backend=/opt/colab-rust/cuda-oxide/lib/librustc_codegen_cuda.so -Zalways-encode-mir -Csymbol-mangling-version=v0
:dep cuda-core = "0.3.1"
:dep cuda-device = { path = "/opt/colab-rust/cuda-oxide/src/crates/cuda-device" }
:dep cuda-host = { path = "/opt/colab-rust/cuda-oxide/src/crates/cuda-host" }
:dep colab-cuda-oxide = { path = "/opt/colab-rust/cuda-oxide/helper" }
println!("cuda-oxide configured for Tesla T4 (sm_75)");


## 4. A kernel, compiled and launched

`#[cuda_module] mod kernels` is compiled to PTX by the cuda-oxide backend and embedded in this cell's build. `load_kernels!` reads it back from there (cuda-oxide's own `kernels::load()` looks in the running executable, which under evcxr is the runtime, not the cell) and hands you the typed launch API. The explicit `kernels::LoadedModule` type lets evcxr keep `loaded` for later cells.


In [ ]:
%%rust
use cuda_core::simt::LaunchConfig;
use cuda_core::{CudaContext, DeviceBuffer};
use cuda_device::{DisjointSlice, cuda_module, kernel, thread};

#[cuda_module]
mod kernels {
    use super::*;

    #[kernel]
    pub fn vecadd(a: &[f32], b: &[f32], mut c: DisjointSlice<f32>) {
        let idx = thread::index_1d();
        let raw = idx.get();
        if let Some(out) = c.get_mut(idx) {
            *out = a[raw] + b[raw];
        }
    }
}

let ctx = CudaContext::new(0).unwrap();
let stream = ctx.default_stream();
let loaded: kernels::LoadedModule = colab_cuda_oxide::load_kernels!(&ctx, kernels).unwrap();

const N: usize = 1024;
let a: Vec<f32> = (0..N).map(|i| i as f32).collect();
let b: Vec<f32> = (0..N).map(|i| (i * 2) as f32).collect();
let a_dev = DeviceBuffer::from_host(&stream, &a).unwrap();
let b_dev = DeviceBuffer::from_host(&stream, &b).unwrap();
let mut c_dev = DeviceBuffer::<f32>::zeroed(&stream, N).unwrap();
unsafe { loaded.vecadd(&stream, LaunchConfig::for_num_elems(N as u32), &a_dev, &b_dev, &mut c_dev) }.unwrap();
println!("{:?}", &c_dev.to_host_vec(&stream).unwrap()[..5]);


## 5. GPU state survives across cells

`ctx`, `stream`, the device buffers and `loaded` all live on in the evcxr runtime, so the next cell can launch again without re-uploading anything.


In [ ]:
%%rust
let mut d_dev = DeviceBuffer::<f32>::zeroed(&stream, N).unwrap();
unsafe { loaded.vecadd(&stream, LaunchConfig::for_num_elems(N as u32), &a_dev, &c_dev, &mut d_dev) }.unwrap();
println!("{:?}", &d_dev.to_host_vec(&stream).unwrap()[..5]);


## Things to know

- **Redefining the module replaces it.** A later cell with its own `#[cuda_module] mod kernels { ... }` swaps in the new kernels; call `load_kernels!` again in that cell to get the new `LoadedModule`.
- **`ThreadIndex` is not `Copy`.** Take `idx.get()` before `c.get_mut(idx)`, as above, or the compiler will complain about a moved value.
- **Annotate `loaded`.** evcxr can only keep a variable across cells when it knows the type name, so write `let loaded: kernels::LoadedModule = ...`. Inside a `{ }` block nothing needs annotating.
- **Stop clears variables.** Pressing stop interrupts the running cell but resets evcxr's variables, including `ctx` and the buffers; items such as `mod kernels` from earlier cells stay. Run cell 4 again to rebuild the state.
- **Back to plain Rust.** `%rust_reset` restarts evcxr on the stable toolchain without the backend; run cell 3 again to return to cuda-oxide.
- **What is pinned.** cuda-oxide commit `6abfaa09` (2026-09-11), `nightly-2026-08-28`, CUDA 13.4, `cuda-core` 0.3.1. cuda-oxide is alpha software; the setup script pins these on purpose and `COLAB_CUDA_OXIDE_REF` overrides the commit.
